# Baseline pipeline: CatBoost + региональный OOF-признак

Один простой, честный baseline (без ансамблей и тюнинга) + региональный
OOF-энкодинг `adminarea` взвешенной медианой. Метрика — **WMAE**, вес `w`
используется как `sample_weight`.


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from src.config import (
    CSV_READ_KWARGS, TRAIN_PATH, TEST_PATH, TARGET_COL, WEIGHT_COL, ID_COL,
    DATE_COL, CATEGORICAL_COLS, REGION_COL, MIN_GROUP_COUNT, N_FOLDS,
    RANDOM_SEED, OUTPUTS_DIR,
)
from src.preprocessing import preprocess
from src.metrics import wmae, weighted_median
from src.validation import get_folds
from src.region_encoding import (
    oof_region_encoding, fit_full_region_encoding, apply_region_encoding,
)
from src.modeling import LogTargetCatBoost, prepare_cat_features

np.random.seed(RANDOM_SEED)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)


## 1. Препроцессинг

In [2]:
train_raw = pd.read_csv(TRAIN_PATH, **CSV_READ_KWARGS, low_memory=False)
test_raw = pd.read_csv(TEST_PATH, **CSV_READ_KWARGS, low_memory=False)

train, train_fixed_cols = preprocess(train_raw, is_train=True)
test, test_fixed_cols = preprocess(test_raw, is_train=False)

print("train:", train.shape, "| test:", test.shape)
print(f"dtype-fixed columns: {len(train_fixed_cols)} (train), {len(test_fixed_cols)} (test)")


train: (76786, 224) | test: (73214, 222)
dtype-fixed columns: 33 (train), 33 (test)


In [3]:
RAW_FEATURE_COLS = [c for c in train.columns if c not in [ID_COL, TARGET_COL, WEIGHT_COL]]
CAT_COLS = list(CATEGORICAL_COLS)  # includes 'dt'

y = train[TARGET_COL].to_numpy()
w = train[WEIGHT_COL].to_numpy()

print(f"raw feature columns: {len(RAW_FEATURE_COLS)}")
print(f"categorical columns (native CatBoost support): {CAT_COLS}")


raw feature columns: 221
categorical columns (native CatBoost support): ['dt', 'gender', 'adminarea', 'city_smart_name', 'dp_ewb_last_employment_position', 'addrref', 'dp_address_unique_regions', 'period_last_act_ad']


`dt` остаётся одной из 8 категориальных колонок для CatBoost (нативная
поддержка категорий, без ручного FE над датами — см. раздел 6 ТЗ, "не
делать сложный feature engineering"). Пропуски в категориальных колонках
заполняются меткой `'missing'` (`prepare_cat_features`), остальные 33
misclassified-числовые уже приведены к float в `preprocess()`.

## 2. Метрика

In [4]:
# Быстрая демонстрация метрики (полное покрытие - в tests/test_metrics.py)
demo_y_true = np.array([10.0, 20.0, 30.0])
demo_y_pred = np.array([12.0, 18.0, 35.0])
demo_w = np.array([1.0, 1.0, 3.0])
print("unweighted MAE:", wmae(demo_y_true, demo_y_pred))
print("WMAE (weighted):", wmae(demo_y_true, demo_y_pred, demo_w))


unweighted MAE: 3.0
WMAE (weighted): 3.8


## 3. Региональный признак (OOF weighted-median encoding)

In [5]:
folds = get_folds(len(train), n_folds=N_FOLDS, seed=RANDOM_SEED)

# OOF-энкодинг считается один раз по этим же 5 фолдам, которыми ниже будет
# оцениваться и сама модель - поэтому кодировка каждой строки в fold i
# зависит только от строк ВНЕ fold i (см. src/region_encoding.py).
region_oof_train = oof_region_encoding(
    train, y, w, folds, region_col=REGION_COL, min_count=MIN_GROUP_COUNT
)
train["region_income_encoding"] = region_oof_train
print("region_income_encoding (train) - sample:")
train[[REGION_COL, "region_income_encoding"]].head()


region_income_encoding (train) - sample:


,adminarea,region_income_encoding
0,Свердловская область,64988.060000
1,Краснодарский край,61856.250000
2,Новосибирская область,60122.775182
3,Хабаровский край,105536.000000
4,Новосибирская область,57352.620000


## 4. Валидация

In [6]:
print(f"K-Fold: {N_FOLDS} folds, shuffle=True, seed={RANDOM_SEED}")
for i, (tr_idx, val_idx) in enumerate(folds):
    print(f"  fold {i}: train={len(tr_idx)}, val={len(val_idx)}")


K-Fold: 5 folds, shuffle=True, seed=42
  fold 0: train=61428, val=15358
  fold 1: train=61429, val=15357
  fold 2: train=61429, val=15357
  fold 3: train=61429, val=15357
  fold 4: train=61429, val=15357


In [7]:
# Раздел 4 ноутбука 1 показал: dt train (2024-01..06) и test (2024-07..11)
# не пересекаются, adversarial AUC~0.85 -> реальный temporal shift.
# Добавляем time-based split: последний месяц train как валидация.
print(train[DATE_COL].value_counts().sort_index())

time_cutoff = train[DATE_COL].max()  # 2024-06-30
time_train_mask = (train[DATE_COL] < time_cutoff).to_numpy()
time_val_mask = (train[DATE_COL] == time_cutoff).to_numpy()
print(f"time-based split: train={time_train_mask.sum()}, val={time_val_mask.sum()} (holdout month: {time_cutoff.date()})")


dt
2024-01-31     7243
2024-02-29     8865
2024-03-31    13413
2024-04-30    14858
2024-05-31    16193
2024-06-30    16214
Name: count, dtype: int64
time-based split: train=60572, val=16214 (holdout month: 2024-06-30)


## 5. Sanity floor: региональный тривиальный baseline

In [8]:
sanity_floor_wmae = wmae(y, region_oof_train, w)
print(f"Sanity floor (OOF region weighted-median baseline) WMAE: {sanity_floor_wmae:,.0f}")
print("Reference from ТЗ section 0.6 (best region baseline): ~120 225")


Sanity floor (OOF region weighted-median baseline) WMAE: 120,225
Reference from ТЗ section 0.6 (best region baseline): ~120 225


Ожидание из ТЗ (раздел 0.6) — около 120 000 — воспроизведено с точностью
до сотен единиц. Полная модель ниже должна **заметно** побить этот
ориентир, иначе остальные ~200 признаков не используются эффективно.

## 6. Модель

In [9]:
def build_feature_matrix(df):
    X = df[RAW_FEATURE_COLS + ["region_income_encoding"]].copy()
    X[DATE_COL] = X[DATE_COL].astype(str)
    X = prepare_cat_features(X, CAT_COLS)
    return X

X_train_full = build_feature_matrix(train)
print("feature matrix shape:", X_train_full.shape)


feature matrix shape: (76786, 222)


In [10]:
fold_wmae = []
fold_best_iters = []
oof_model_preds = np.full(len(train), np.nan)

for i, (tr_idx, val_idx) in enumerate(folds):
    model = LogTargetCatBoost(cat_features=CAT_COLS)
    model.fit(
        X_train_full.iloc[tr_idx], y[tr_idx], sample_weight=w[tr_idx],
        eval_set=(X_train_full.iloc[val_idx], y[val_idx], w[val_idx]),
    )
    preds = model.predict(X_train_full.iloc[val_idx])
    oof_model_preds[val_idx] = preds

    fold_score = wmae(y[val_idx], preds, w[val_idx])
    fold_wmae.append(fold_score)
    fold_best_iters.append(model.model.get_best_iteration())
    print(f"fold {i}: WMAE={fold_score:,.0f}  (best_iteration={fold_best_iters[-1]})")

cv_wmae_mean = np.mean(fold_wmae)
cv_wmae_std = np.std(fold_wmae)
pooled_oof_wmae = wmae(y, oof_model_preds, w)

print(f"\nCV WMAE: {cv_wmae_mean:,.0f} +/- {cv_wmae_std:,.0f}")
print(f"Pooled OOF WMAE: {pooled_oof_wmae:,.0f}")
print(f"Sanity floor (region-only):  {sanity_floor_wmae:,.0f}")
print(f"Improvement over sanity floor: {100 * (1 - cv_wmae_mean / sanity_floor_wmae):.1f}%")


fold 0: WMAE=74,523  (best_iteration=1999)


fold 1: WMAE=68,299  (best_iteration=1647)


fold 2: WMAE=71,598  (best_iteration=1998)


fold 3: WMAE=68,173  (best_iteration=1489)


fold 4: WMAE=72,418  (best_iteration=1987)

CV WMAE: 71,002 +/- 2,452
Pooled OOF WMAE: 71,031
Sanity floor (region-only):  120,225
Improvement over sanity floor: 40.9%


In [11]:
# Time-based validation (see section 4): region encoding refit on the
# earlier months only, to avoid leaking the held-out month into the encoding.
from src.region_encoding import compute_region_stats, apply_region_stats

time_stats, time_fallback = compute_region_stats(
    train.loc[time_train_mask, REGION_COL].to_numpy(),
    y[time_train_mask], w[time_train_mask], min_count=MIN_GROUP_COUNT,
)
# stats were fit only on the pre-cutoff rows, so applying them to every
# row (train and held-out val) is leakage-safe by construction.
region_time_encoding = apply_region_stats(train[REGION_COL].to_numpy(), time_stats, time_fallback)
train_time = train.copy()
train_time["region_income_encoding"] = region_time_encoding
X_time = build_feature_matrix(train_time)

model_time = LogTargetCatBoost(cat_features=CAT_COLS)
model_time.fit(
    X_time.iloc[time_train_mask], y[time_train_mask], sample_weight=w[time_train_mask],
    eval_set=(X_time.iloc[time_val_mask], y[time_val_mask], w[time_val_mask]),
)
time_preds = model_time.predict(X_time.iloc[time_val_mask])
time_wmae = wmae(y[time_val_mask], time_preds, w[time_val_mask])

print(f"Time-based holdout WMAE (train<{time_cutoff.date()} -> val={time_cutoff.date()}): {time_wmae:,.0f}")
print(f"Random 5-fold CV WMAE (comparison): {cv_wmae_mean:,.0f} +/- {cv_wmae_std:,.0f}")


Time-based holdout WMAE (train<2024-06-30 -> val=2024-06-30): 69,217
Random 5-fold CV WMAE (comparison): 71,002 +/- 2,452


**Вывод раздела 6**: CV WMAE заметно лучше sanity floor — модель
эффективно использует признаки за пределами регионального. Time-based
holdout сравнивается со случайным K-Fold: если time-based WMAE заметно хуже,
это подтверждает temporal shift из ноутбука 1 и означает, что реальное
качество на приватном тесте, скорее всего, ближе к time-based оценке, чем
к случайному CV.

## 7. Интерпретация

In [12]:
# Финальная модель: фиксируем число итераций по средней best_iteration
# из CV, обучаем на ВСЕХ данных train без early stopping (иначе early
# stopping требовал бы утечки за счёт eval на части train).
final_iterations = int(np.mean(fold_best_iters)) + 1
print(f"final_iterations (avg best_iteration across folds + 1): {final_iterations}")

full_stats, full_fallback = fit_full_region_encoding(train, y, w, region_col=REGION_COL, min_count=MIN_GROUP_COUNT)
train["region_income_encoding"] = apply_region_encoding(train, full_stats, full_fallback, region_col=REGION_COL)
X_train_final = build_feature_matrix(train)

final_model = LogTargetCatBoost(
    cat_features=CAT_COLS,
    params={"iterations": final_iterations, "early_stopping_rounds": None},
)
final_model.fit(X_train_final, y, sample_weight=w)
print("final model trained on 100% of train.")


final_iterations (avg best_iteration across folds + 1): 1825


final model trained on 100% of train.


In [13]:
feature_importance = final_model.get_feature_importance()
feature_importance.head(20).to_frame("importance").to_csv(OUTPUTS_DIR / "feature_importance.csv")
region_rank = list(feature_importance.index).index("region_income_encoding") + 1
print(f"region_income_encoding rank: {region_rank} / {len(feature_importance)}")
feature_importance.head(20)


region_income_encoding rank: 21 / 222


salary_6to12m_avg                                                                   13.959807
turn_cur_cr_avg_act_v2                                                               8.812189
hdb_bki_total_max_limit                                                              5.474098
gender                                                                               4.338819
incomeValue                                                                          3.246214
turn_cur_db_avg_act_v2                                                               3.204328
hdb_bki_total_cc_max_limit                                                           2.953477
dp_ils_paymentssum_avg_12m                                                           2.924149
hdb_bki_total_pil_max_limit                                                          2.869372
turn_cur_cr_avg_v2                                                                   2.141219
avg_cur_cr_turn                                             

**Вывод раздела 7**: `outputs/feature_importance.csv` сохранён (топ-20).
Место `region_income_encoding` печатается явно выше — сравнить с рангом
"зарплатных" признаков (`salary_6to12m_avg`,
`dp_payoutincomedata_payout_avg_3_month` и т.п.) из раздела 0.5/0.6 ТЗ:
региональный признак — легитимный, но умеренный по силе сигнал, как и
предсказывал раздел 0.6 (region-only объясняет лишь часть дисперсии).

## 8. Сабмит

In [14]:
test["region_income_encoding"] = apply_region_encoding(test, full_stats, full_fallback, region_col=REGION_COL)
X_test_final = build_feature_matrix(test)

test_preds = final_model.predict(X_test_final)

submission = pd.DataFrame({ID_COL: test[ID_COL], TARGET_COL: test_preds})
submission.to_csv(OUTPUTS_DIR / "submission_baseline.csv", index=False)
print("saved:", OUTPUTS_DIR / "submission_baseline.csv")
submission.head()


saved: /Users/nikitabaslykov/Documents/Python/ML/Client_Analytics/outputs/submission_baseline.csv


,id,target
0,0,63214.493166
1,1,58169.385237
2,3,37001.065496
3,9,80959.160013
4,11,47102.585331


In [15]:
# Sanity-check перед отправкой
assert submission[TARGET_COL].isna().sum() == 0, "NaN predictions found"
assert (submission[TARGET_COL] >= 0).all(), "negative predictions found"
assert len(submission) == len(test), "row count mismatch vs test.csv"

train_target_range = (train[TARGET_COL].min(), train[TARGET_COL].max())
pred_range = (submission[TARGET_COL].min(), submission[TARGET_COL].max())
print("train target range:", train_target_range)
print("prediction range:  ", pred_range)
print("all sanity checks passed.")


train target range: (np.float64(20000.0), np.float64(1500000.0))
prediction range:   (np.float64(16921.305649943697), np.float64(1618506.14039528))
all sanity checks passed.


**Допущение (нужно подтвердить у организаторов)**: `sample_submission.csv`
в данных не обнаружен, поэтому формат сабмита выбран как `id,target`
(разделитель `,`, точка как десятичный разделитель) — см. README.md.

## Итоговая сводка (ноутбук 2)

- CV WMAE (5-fold, случайный сплит): печатается в разделе 6 выше, сравнение
  со sanity floor (региональный weighted-median baseline, раздел 5).
- Time-based holdout (последний месяц train) даёт независимую оценку под
  temporal shift, обнаруженный в ноутбуке 1.
- Финальная модель обучена на 100% train с региональным OOF-признаком,
  число итераций зафиксировано по средней `best_iteration` из CV (без
  дополнительного тюнинга, без ансамблей — см. раздел 6 ТЗ).
- `outputs/submission_baseline.csv`, `outputs/feature_importance.csv` и
  `outputs/feature_stats.csv` (из ноутбука 1) сохранены для дальнейшей
  работы.
